In [ ]:
# Data file: set CMSOPEN_DATA_FILE to the absolute path of a NanoAOD ROOT file
# before starting Jupyter, e.g. the 2012C DoubleMuParked file described in DATA.md.
import os
from pathlib import Path

if not os.environ.get("CMSOPEN_DATA_FILE"):
    raise RuntimeError(
        "CMSOPEN_DATA_FILE is not set. Start Jupyter with "
        "CMSOPEN_DATA_FILE=/absolute/path/to/file.root jupyter lab ..."
    )
DATA_FILE = Path(os.environ["CMSOPEN_DATA_FILE"])
if not DATA_FILE.exists():
    raise FileNotFoundError(f"CMSOPEN_DATA_FILE points to a missing file: {DATA_FILE}")
print(f"Data file: {DATA_FILE}")

# Layer 5: a first look at the Z peak with `cmsopen.kinematics`

Four-momentum is conserved in a decay, so the summed four-vector of the two muons from $Z\to\mu^+\mu^-$ is the Z four-vector. Its Minkowski length, the invariant mass

$$M^2 = (E_1+E_2)^2 - |\vec p_1 + \vec p_2|^2 \quad (c = 1),$$

is the same in every frame and equals the parent's rest mass. Muon pairs from Z decays therefore pile up near 91.19 GeV, while background pairs spread smoothly.

**This is not the final analysis.** Selection (layer 3) and pairing (layer 4) do not exist yet, so a simple selection is written here with plain awkward masks: at least two muons; the two highest-$p_T$ muons have opposite charge, $p_T > 20$ GeV and $|\eta| < 2.4$.

In [ ]:
import awkward as ak
import matplotlib.pyplot as plt
import numpy as np

import cmsopen
from cmsopen import FourMomentum, Muon, invariant_mass, open_events

try:
    import vector
except ImportError:
    vector = None

print(f"cmsopen imported from: {cmsopen.__file__}")
print(f"vector library: {vector.__version__ if vector else 'not installed (pip install vector to enable the cross-check)'}")

## Select the leading muon pair in each event

In [ ]:
PT_MIN = 20.0  # GeV
ETA_MAX = 2.4

with open_events(DATA_FILE) as events:
    n_total = len(events)
    muons = Muon.from_events(events, fields=("pt", "eta", "phi", "mass", "charge"))

muons = muons[ak.argsort(muons.p4.pt, axis=1, ascending=False)]
muons = muons[muons.num >= 2]
lead, sub = muons[:, 0], muons[:, 1]
selected = (
    (lead.charge * sub.charge < 0)
    & (lead.p4.pt > PT_MIN) & (sub.p4.pt > PT_MIN)
    & (np.abs(lead.p4.eta) < ETA_MAX) & (np.abs(sub.p4.eta) < ETA_MAX)
)
lead, sub = lead[selected], sub[selected]
mass = ak.to_numpy(invariant_mass(lead.p4, sub.p4))

print(f"events in file:                         {n_total:,}")
print(f"events with >= 2 muons:                 {len(muons):,}")
print(f"events used (leading pair passes cuts): {len(mass):,}")

## Text histogram, 60 to 120 GeV in 2 GeV bins

In [ ]:
BINS = np.arange(60.0, 122.0, 2.0)
counts, edges = np.histogram(mass, bins=BINS)
print(f"pairs with 60 <= M < 120 GeV: {int(counts.sum()):,}")
print("Dimuon invariant mass (GeV), 2 GeV bins:")
scale = 50 / counts.max() if counts.max() else 0
for lo, hi, n in zip(edges[:-1], edges[1:], counts):
    print(f"  {lo:5.0f}-{hi:<5.0f} {n:7d} {'#' * int(round(n * scale))}")

## The same spectrum with matplotlib

A plain histogram; publication styling comes in layer 7. The peak width combines the Z's natural width (2.4952 GeV) with the detector's momentum resolution.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(mass, bins=BINS, histtype="step", linewidth=1.5, label="opposite-charge leading muon pairs")
ax.axvline(91.1876, color="gray", linestyle="--", linewidth=1, label="PDG Z mass 91.1876 GeV")
ax.set_xlabel("Dimuon invariant mass $M_{\\mu\\mu}$ [GeV]")
ax.set_ylabel("Pairs / 2 GeV")
ax.set_title("CMS Open Data 2012C DoubleMuParked (first look)")
ax.legend()
plt.show()

## Self-checks

1. All masses are finite and non-negative.
2. The highest bin between 70 and 110 GeV lies within 88 to 94 GeV: a Z peak is visible.
3. A Z at rest decaying to two back-to-back muons gives back 91.1876 GeV.
4. The masses agree with the independent `vector` library on the real pairs, if installed.

In [ ]:
class Checks:
    """Collects self-check results and prints one OK/FAIL/SKIP line each."""

    def __init__(self):
        self.failures = []

    def check(self, name, ok, detail):
        print(f"{'OK' if ok else 'FAIL'}: {name} ({detail})")
        if not ok:
            self.failures.append(name)

    def skip(self, name, reason):
        print(f"SKIP: {name} ({reason})")

In [ ]:
Z_MASS = 91.1876
MU_MASS = 0.10566
PEAK_SEARCH = (70.0, 110.0)
PEAK_WINDOW = (88.0, 94.0)

checks = Checks()
checks.check(
    "all masses finite and non-negative",
    bool(np.all(np.isfinite(mass)) and np.all(mass >= 0)),
    f"{len(mass):,} masses, min {mass.min():.4g} GeV, max {mass.max():.4g} GeV" if len(mass) else "no pairs",
)

in_search = (edges[:-1] >= PEAK_SEARCH[0]) & (edges[1:] <= PEAK_SEARCH[1])
peak = int(np.flatnonzero(in_search)[np.argmax(counts[in_search])])
peak_lo, peak_hi = edges[peak], edges[peak + 1]
checks.check(
    f"Z peak: highest bin in {PEAK_SEARCH[0]:.0f}-{PEAK_SEARCH[1]:.0f} GeV lies within "
    f"{PEAK_WINDOW[0]:.0f}-{PEAK_WINDOW[1]:.0f} GeV",
    bool(peak_lo >= PEAK_WINDOW[0] and peak_hi <= PEAK_WINDOW[1]),
    f"peak bin {peak_lo:.0f}-{peak_hi:.0f} GeV with {counts[peak]:,} pairs",
)

p = np.sqrt((Z_MASS / 2) ** 2 - MU_MASS**2)  # two-body decay momentum
leg1 = FourMomentum.from_arrays([p], [0.0], [0.0], [MU_MASS])
leg2 = FourMomentum.from_arrays([p], [0.0], [np.pi], [MU_MASS])
m_rest = float(invariant_mass(leg1, leg2)[0])
checks.check(
    "Z at rest -> mu+ mu- gives back the Z mass",
    abs(m_rest - Z_MASS) <= 1e-9 * Z_MASS,
    f"{m_rest:.10f} GeV, expected {Z_MASS}",
)

if vector is None:
    checks.skip("agreement with vector library on the real pairs", "vector is not installed")
else:
    v1, v2 = (vector.zip({"pt": q.pt, "eta": q.eta, "phi": q.phi, "mass": q.mass}) for q in (lead.p4, sub.p4))
    theirs = ak.to_numpy((v1 + v2).mass)
    rel = np.abs(mass - theirs) / np.maximum(theirs, 1e-300)
    checks.check(
        f"agreement with vector {vector.__version__} on all {len(mass):,} real pairs",
        bool(np.allclose(mass, theirs, rtol=1e-9, atol=1e-9)),
        f"max relative difference {rel.max():.3g}",
    )

if checks.failures:
    raise AssertionError(f"{len(checks.failures)} check(s) failed: {', '.join(checks.failures)}")
print("All checks passed")